In [1]:
import sys
print(sys.executable)

/mnt/beegfs/home/yyu2024/my_pytorch_env/bin/python


In [15]:
!{sys.executable} -m pip install numpy mne scipy plotly pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 61.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [3]:
import os
import zipfile
import urllib.request
import pathlib

# URLs for the ZIP files
zip_url_1 = "https://www.bbci.de/competition/download/competition_iv/BCICIV_4_mat.zip"
zip_url_2 = "https://www.bbci.de/competition/iv/results/ds4/true_labels.zip"

# Directory where you want to save the files
data_dir = pathlib.Path().resolve() / "data" / "pure_data"

# Ensure the directory exists, create it if it doesn't
os.makedirs(data_dir, exist_ok=True)

# Paths to the ZIP files and the .mat files
zip_file_path_1 = data_dir / "BCICIV_4_mat.zip"
mat_file_path_1 = data_dir / "sub1_comp.mat"

zip_file_path_2 = data_dir / "true_labels.zip"
mat_file_path_2 = data_dir / "sub1_testlabels.mat"

# Function to download the ZIP file if needed and always extract the .mat file
def download_and_extract(zip_url, zip_file_path, mat_filename, mat_file_path):
    if not zip_file_path.exists():
        # Download the ZIP file if it doesn't exist
        print(f"Downloading {zip_file_path}...")
        urllib.request.urlretrieve(zip_url, zip_file_path)
    else:
        print(f"ZIP file {zip_file_path} already exists, skipping download.")

    # Always extract the mat file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extract(mat_filename, data_dir)
    print(f"Extracted {mat_filename} to: {mat_file_path}")

# Download and extract for the first ZIP
download_and_extract(zip_url_1, zip_file_path_1, 'sub1_comp.mat', mat_file_path_1)

# Download and extract for the second ZIP
download_and_extract(zip_url_2, zip_file_path_2, 'sub1_testlabels.mat', mat_file_path_2)


ZIP file /mnt/onefs/home/yyu2024/PLaCT/FingerFlex/data/pure_data/BCICIV_4_mat.zip already exists, skipping download.
Extracted sub1_comp.mat to: /mnt/onefs/home/yyu2024/PLaCT/FingerFlex/data/pure_data/sub1_comp.mat
ZIP file /mnt/onefs/home/yyu2024/PLaCT/FingerFlex/data/pure_data/true_labels.zip already exists, skipping download.
Extracted sub1_testlabels.mat to: /mnt/onefs/home/yyu2024/PLaCT/FingerFlex/data/pure_data/sub1_testlabels.mat


In [ ]:
import numpy as np
import mne
import scipy.io, scipy.interpolate
import plotly.express as px
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# mne.utils.set_config('MNE_USE_CUDA', 'true')

L_FREQ, H_FREQ = 40, 300 # Lower and upper filtration bounds
CHANNELS_NUM = 62        # Number of channels in ECoG data
WAVELET_NUM = 40         # Number of wavelets in the indicated frequency range, with which the convolution is performed
DOWNSAMPLE_FS = 100      # Desired sampling rate
time_delay_secs = 0.2    # Time delay hyperparameter

current_fs = DOWNSAMPLE_FS

# =============================================================================
# PREPROCESSING MODE SELECTION
# =============================================================================
# Set to 'fingerflex' for original FingerFlex preprocessing (wavelets)
# Set to 'bc4d4' for BC4D4 preprocessing (Isolation Forest, raw signals)
PREPROCESSING_MODE = 'bc4d4'  # <-- CHANGE THIS TO SWITCH METHODS

FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Little']

# =============================================================================
# FINGERFLEX PREPROCESSING FUNCTIONS (Original)
# =============================================================================

def reshape_column_ecog_data(multichannel_signal: np.ndarray):
    return multichannel_signal.T # (time, features) -> (features, time)

def filter_ecog_data(multichannel_signal: np.ndarray, fs=1000, powerline_freq=50):
    """
    Harmonics removal and frequency filtering
    :param multichannel_signal: Initial multi-channel signal
    :param fs: Sampling rate
    :param powerline_freq: Grid frequency
    :return: Filtered signal
    """
    harmonics = np.array([i * powerline_freq for i in range(1, (fs // 2) // powerline_freq)])

    print("Starting...")
    signal_filtered = mne.filter.filter_data(multichannel_signal,
                                             fs, l_freq=L_FREQ, h_freq=H_FREQ)  # remove all frequencies between l and h
    print("Noise frequencies removed...")
    signal_removed_powerline_noise = mne.filter.notch_filter(signal_filtered,
                                                             fs, freqs=harmonics)  # remove powerline  noise
    print("Powerline noise removed...")
    
    return signal_removed_powerline_noise

def normalize(multichannel_signal: np.ndarray, return_values = None):
    """
    standardization and removal of the median  from each channel
    :param multichannel_signal: Multi-channel signal
    :param return_values: Whether to return standardization parameters. By default - no
    """
    print("Normalizing...")
    means = np.mean(multichannel_signal, axis=1, keepdims=True)
    stds = np.std(multichannel_signal, axis=1, keepdims=True)
    transformed_data = (multichannel_signal - means) / stds
    common_average = np.median(transformed_data, axis=0, keepdims=True)
    transformed_data = transformed_data - common_average
    if return_values:
        return transformed_data, (means, stds)
    print("Normalized...")
    return transformed_data

def compute_spectrogramms(multichannel_signal : np.ndarray, fs=1000, freqs=np.logspace(np.log10(L_FREQ), np.log10(H_FREQ), WAVELET_NUM),
                          output_type='power'):
    """
    Compute spectrogramms using wavelet transforms

    :param freqs: wavelet frequencies to uses
    :param fs: Sampling rate
    :return: Signal spectogramms in shape (channels, wavelets, time)
    """
    
    num_of_channels = len(multichannel_signal)

    print("Computing wavelets...")
    spectrogramms = mne.time_frequency.tfr_array_morlet(multichannel_signal.reshape(1, num_of_channels, -1), sfreq=fs,
                                                        freqs=freqs, output=output_type, verbose=10, n_jobs=6)[0]
    
    
    print("Wavelet spectrogramm computed...")
    
    return spectrogramms


def downsample_spectrogramms(spectrogramms: np.ndarray, cur_fs=1000, needed_hz=H_FREQ, new_fs = None):
    """
    Reducing the sampling rate of spectrograms
    :param spectrogramms: Original set of spectrograms
    :param cur_fs: Current sampling rate
    :param needed_hz: The maximum frequency that must be unambiguously preserved during compression
    :param new_fs: The required sampling rate (interchangeable with needed_hz)
    :return: Decimated signal
    """
    print("Downsampling spectrogramm...")
    if new_fs == None:
        new_fs = needed_hz * 2    
    downsampling_coef = cur_fs // new_fs
    assert downsampling_coef > 1
    downsampled_spectrogramm = spectrogramms[:, :, ::downsampling_coef]
    print("Spectrogramm downsampled...")
    return downsampled_spectrogramm


def normalize_spectrogramms_to_db(spectrogramms: np.ndarray, convert = False):
    """
    Optional conversion to db, not used in the final version
    """
    if convert:
        return np.log10(spectrogramms+1e-12)
    else:
        return spectrogramms


def interpolate_fingerflex(finger_flex, cur_fs=1000, true_fs=25, needed_hz=DOWNSAMPLE_FS, interp_type='cubic'):
    
    """
    Interpolation of the finger motion recording to match the new sampling rate
    :param finger_flex: Initial sequences with finger flexions data
    :param cur_fs: ECoG sampling rate
    :param true_fs: Actual finger motions recording sampling rate
    :param needed_hz: Required sampling rate
    :param interp_type: Type of interpolation. By default - cubic
    :return: Returns an interpolated set of finger motions with the desired sampling rate
    """
    
    print("Interpolating fingerflex...")
    downscaling_ratio = cur_fs // true_fs
    print("Computing true_fs values...")
    finger_flex_true_fs = finger_flex[:, ::downscaling_ratio]
    finger_flex_true_fs = np.c_[finger_flex_true_fs,
        finger_flex_true_fs.T[-1]]  # Add as the last value on the interpolation edge the last recorded
    # Because otherwise it is not clear how to interpolate the tail at the end

    upscaling_ratio = needed_hz // true_fs
    
    ts = np.asarray(range(finger_flex_true_fs.shape[1])) * upscaling_ratio
    
    print("Making funcs...")
    interpolated_finger_flex_funcs = [scipy.interpolate.interp1d(ts, finger_flex_true_fs_ch, kind=interp_type) for
                                     finger_flex_true_fs_ch in finger_flex_true_fs]
    ts_needed_hz = np.asarray(range(finger_flex_true_fs.shape[1] * upscaling_ratio)[
                              :-upscaling_ratio])  # Removing the extra added edge
    
    print("Interpolating with needed frequency")
    interpolated_finger_flex = np.array([[interpolated_finger_flex_func(t) for t in ts_needed_hz] for
                                         interpolated_finger_flex_func in interpolated_finger_flex_funcs])
    return interpolated_finger_flex


def crop_for_time_delay(finger_flex : np.ndarray, spectrogramms : np.ndarray, time_delay_sec : float, fs : int):
    """
    Taking into account the delay between brain waves and movements
    :param finger_flex: Finger flexions
    :param spectrogramms: Computed spectrogramms
    :param time_delay_sec: time delay hyperparameter
    :param fs: Sampling rate
    :return: Shifted series with a delay
    """

    time_delay = int(time_delay_sec*fs)

    # the first motions do not depend on available data
    finger_flex_cropped = finger_flex[..., time_delay:] 
    # The latter spectrograms have no corresponding data
    spectrogramms_cropped = spectrogramms[..., :spectrogramms.shape[2]-time_delay]
    return finger_flex_cropped, spectrogramms_cropped


def visualize_signal(multichannel_signal: np.ndarray, channel_num: int, second_num: int, fs=DOWNSAMPLE_FS):
    """
    Function to visualize multi-channel signal section
    :param multichannel_signal: Multi-channel signal
    :param channel_num: Channel selected for visualization
    :param second_num: Selected record second
    :param fs: Sampling rate
    :return: -
    """
    df_channel = pd.DataFrame(data=np.asarray([np.asarray(range(fs)),
                                               multichannel_signal[channel_num][second_num*fs:second_num*fs+fs]]).T,
                              index=range(fs), columns=["t", "V"])

    fig = px.line(df_channel, x="t", y="V", title=f'channel_{channel_num}')
    fig.show()


# =============================================================================
# BC4D4 PREPROCESSING FUNCTIONS (Jangir et al. 2025)
# =============================================================================
# Key differences from FingerFlex:
# 1. Uses Isolation Forest for outlier removal
# 2. Works on RAW ECoG signals (no wavelet spectrograms)
# 3. Transforms data to near-Gaussian distribution [-1, +1]

def compute_descriptive_stats_bc4d4(data: np.ndarray):
    """
    Compute descriptive statistics for finger movement data.
    Replicates Table 2 from BC4D4 paper.
    """
    # Ensure shape is (time, 5)
    if data.shape[0] == 5:
        data = data.T
    
    df = pd.DataFrame(data, columns=FINGER_NAMES)
    stats = df.describe().T.round(2)
    return stats


def plot_box_plots_bc4d4(data: np.ndarray, title: str = "Finger Movement Distribution"):
    """
    Create box plots for finger movement data.
    Used to identify outliers before/after Isolation Forest.
    """
    import matplotlib.pyplot as plt
    
    # Ensure shape is (time, 5)
    if data.shape[0] == 5:
        data = data.T
    
    fig, ax = plt.subplots(figsize=(10, 6))
    df = pd.DataFrame(data, columns=FINGER_NAMES)
    df.boxplot(ax=ax)
    ax.set_title(title)
    ax.set_ylabel('Finger Flexion Value')
    ax.set_xlabel('Finger')
    plt.tight_layout()
    return fig, ax


def apply_isolation_forest_bc4d4(data: np.ndarray, contamination='auto', 
                                  n_estimators=100, random_state=42, verbose=True):
    """
    Apply Isolation Forest to remove outliers from finger movement data.
    
    This is the KEY INNOVATION from BC4D4 paper.
    
    After applying Isolation Forest, data should:
    - Have near-Gaussian distribution
    - Fall approximately in range [-1, +1]
    - Have minimal outliers
    
    Parameters
    ----------
    data : np.ndarray
        Finger movement data, shape (time, 5) or (5, time)
    contamination : float or 'auto'
        Expected proportion of outliers
    
    Returns
    -------
    tuple: (cleaned_data, inlier_mask)
    """
    # Ensure shape is (time, 5)
    if data.shape[0] == 5:
        data = data.T
        was_transposed = True
    else:
        was_transposed = False
    
    if verbose:
        print(f"Applying Isolation Forest...")
        print(f"  Input shape: {data.shape}")
        print(f"  Contamination: {contamination}")
    
    iso_forest = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1
    )
    
    # Fit and predict: -1 = outlier, 1 = inlier
    predictions = iso_forest.fit_predict(data)
    inlier_mask = predictions == 1
    
    cleaned_data = data[inlier_mask]
    
    n_outliers = (~inlier_mask).sum()
    n_total = len(data)
    
    if verbose:
        print(f"  Outliers removed: {n_outliers} ({100*n_outliers/n_total:.2f}%)")
        print(f"  Remaining samples: {len(cleaned_data)}")
    
    if was_transposed:
        cleaned_data = cleaned_data.T
    
    return cleaned_data, inlier_mask


def normalize_ecog_zscore_bc4d4(ecog_data: np.ndarray, verbose=True):
    """
    Apply z-score normalization to ECoG data.
    BC4D4 uses simple z-score, not bandpass + wavelet like FingerFlex.
    """
    if verbose:
        print("Applying z-score normalization to ECoG data...")
    
    scaler = StandardScaler()
    normalized = scaler.fit_transform(ecog_data)
    
    if verbose:
        print(f"  Mean: {normalized.mean():.6f}, Std: {normalized.std():.6f}")
    
    return normalized, scaler


def prepare_bc4d4_input(ecog_data: np.ndarray, verbose=True):
    """
    Prepare ECoG data for BC4D4 model input.
    
    BC4D4 Input Format: (N_samples, N_electrodes, 1)
    This is RAW ECoG signals, NOT wavelet spectrograms.
    """
    if verbose:
        print(f"Preparing BC4D4 input format...")
        print(f"  Input shape: {ecog_data.shape}")
    
    # ecog_data shape: (time, electrodes)
    # BC4D4 expects: (time, electrodes, 1)
    bc4d4_input = ecog_data[:, :, np.newaxis]
    
    if verbose:
        print(f"  Output shape: {bc4d4_input.shape}")
    
    return bc4d4_input


print(f"Preprocessing mode: {PREPROCESSING_MODE}")
print("Functions loaded successfully!")

In [ ]:
"""
Loading the raw training data and applying the processing algorithm
Based on PREPROCESSING_MODE: 'fingerflex' or 'bc4d4'
"""

import pathlib

PATH = f"{pathlib.Path().resolve()}/data/pure_data/"

data = scipy.io.loadmat(f'{PATH}/sub1_comp.mat')

if PREPROCESSING_MODE == 'fingerflex':
    # =============================================================================
    # FINGERFLEX PREPROCESSING (Original - wavelets)
    # =============================================================================
    print("=" * 60)
    print("Using FINGERFLEX preprocessing (wavelet spectrograms)")
    print("=" * 60)
    
    interpolated_finger_flex = interpolate_fingerflex(finger_flex=
                               reshape_column_ecog_data(data['train_dg'].astype('float64')))

    db_spectrogramms = normalize_spectrogramms_to_db(spectrogramms=
                       downsample_spectrogramms(spectrogramms=
                       compute_spectrogramms(multichannel_signal=
                       filter_ecog_data(multichannel_signal=
                       normalize(multichannel_signal=
                       reshape_column_ecog_data(data['train_data'].astype('float64'))))), new_fs = DOWNSAMPLE_FS))
    
    ecog_processed = db_spectrogramms
    finger_processed = interpolated_finger_flex

elif PREPROCESSING_MODE == 'bc4d4':
    # =============================================================================
    # BC4D4 PREPROCESSING (Simplified - no sample removal)
    # =============================================================================
    # KEY CHANGE: We do NOT remove samples with Isolation Forest.
    # Instead, we just apply z-score normalization to ECoG data.
    # This ensures train/val have the same number of time points and no distribution shift.
    
    print("=" * 60)
    print("Using BC4D4 preprocessing (z-score normalized raw signals)")
    print("=" * 60)
    
    # Load raw data (time, features) format
    train_ecog_raw = data['train_data'].astype('float64')  # (time, electrodes)
    train_finger_raw = data['train_dg'].astype('float64')  # (time, 5 fingers)
    
    print(f"\nRaw data shapes:")
    print(f"  ECoG: {train_ecog_raw.shape}")
    print(f"  Finger: {train_finger_raw.shape}")
    
    # Step 1: Show statistics before preprocessing
    print("\n[BEFORE preprocessing] Finger statistics:")
    print(compute_descriptive_stats_bc4d4(train_finger_raw))
    
    # Step 2: Z-score normalize ECoG data (fit scaler for later use on validation)
    print("\n[Preprocessing] Z-score normalizing ECoG...")
    ecog_scaler = StandardScaler()
    train_ecog_norm = ecog_scaler.fit_transform(train_ecog_raw)
    print(f"  ECoG normalized - mean: {train_ecog_norm.mean():.6f}, std: {train_ecog_norm.std():.6f}")
    
    # Step 3: Prepare BC4D4 format: (time, electrodes, 1)
    ecog_bc4d4 = train_ecog_norm[:, :, np.newaxis]
    
    # Step 4: Finger data - ensure shape is (time, 5)
    finger_data = train_finger_raw if train_finger_raw.shape[1] == 5 else train_finger_raw.T
    
    ecog_processed = ecog_bc4d4  # Shape: (time, electrodes, 1)
    finger_processed = finger_data  # Shape: (time, 5)
    
    print(f"\nProcessed data shapes:")
    print(f"  ECoG (BC4D4 format): {ecog_processed.shape}")
    print(f"  Finger: {finger_processed.shape}")
    print(f"  Finger range: [{finger_processed.min():.3f}, {finger_processed.max():.3f}]")

else:
    raise ValueError(f"Unknown PREPROCESSING_MODE: {PREPROCESSING_MODE}. Use 'fingerflex' or 'bc4d4'")

In [ ]:
"""
Loading the raw validation data and applying the processing algorithm
Based on PREPROCESSING_MODE: 'fingerflex' or 'bc4d4'

IMPORTANT FOR BC4D4: We do NOT apply Isolation Forest to validation data.
We only apply the same z-score normalization (using training scaler).
This prevents train/val distribution mismatch.
"""

data_2 = scipy.io.loadmat(f'{PATH}/sub1_testlabels.mat')

if PREPROCESSING_MODE == 'fingerflex':
    # =============================================================================
    # FINGERFLEX PREPROCESSING (Original - wavelets)
    # =============================================================================
    interpolated_finger_flex_val = interpolate_fingerflex(finger_flex=
                                                          reshape_column_ecog_data(data_2['test_dg'].astype('float64')))

    db_spectrogramms_val = normalize_spectrogramms_to_db(spectrogramms=
                           downsample_spectrogramms(spectrogramms=
                           compute_spectrogramms(multichannel_signal=
                           filter_ecog_data(multichannel_signal=
                           normalize(multichannel_signal=
                           reshape_column_ecog_data(data['test_data'].astype('float64'))))), new_fs=DOWNSAMPLE_FS))
    
    ecog_processed_val = db_spectrogramms_val
    finger_processed_val = interpolated_finger_flex_val

elif PREPROCESSING_MODE == 'bc4d4':
    # =============================================================================
    # BC4D4 PREPROCESSING - VALIDATION
    # =============================================================================
    # KEY FIX: Do NOT apply Isolation Forest to validation data!
    # Only apply the same z-score normalization using the TRAINING scaler.
    # This ensures train/val have consistent preprocessing.
    
    print("\n" + "=" * 60)
    print("Processing VALIDATION data with BC4D4...")
    print("NOTE: NOT applying Isolation Forest to validation (prevents distribution shift)")
    print("=" * 60)
    
    # Load raw validation data
    val_ecog_raw = data['test_data'].astype('float64')  # (time, electrodes)
    val_finger_raw = data_2['test_dg'].astype('float64')  # (time, 5 fingers)
    
    print(f"\nRaw validation data shapes:")
    print(f"  ECoG: {val_ecog_raw.shape}")
    print(f"  Finger: {val_finger_raw.shape}")
    
    # Z-score normalize using TRAINING scaler (ecog_scaler from training step)
    print("\nZ-score normalizing validation ECoG using TRAINING scaler...")
    val_ecog_norm = ecog_scaler.transform(val_ecog_raw)
    print(f"  Val ECoG normalized - mean: {val_ecog_norm.mean():.4f}, std: {val_ecog_norm.std():.4f}")
    
    # Prepare BC4D4 format: (time, electrodes, 1)
    ecog_bc4d4_val = val_ecog_norm[:, :, np.newaxis]
    
    # Finger data: just ensure shape is (time, 5), no Isolation Forest
    finger_val = val_finger_raw if val_finger_raw.shape[1] == 5 else val_finger_raw.T
    
    ecog_processed_val = ecog_bc4d4_val
    finger_processed_val = finger_val
    
    print(f"\nProcessed validation data shapes:")
    print(f"  ECoG (BC4D4 format): {ecog_processed_val.shape}")
    print(f"  Finger: {finger_processed_val.shape}")
    print(f"  Finger range: [{finger_processed_val.min():.3f}, {finger_processed_val.max():.3f}]")

In [ ]:
"""
Taking time delay into account (FingerFlex only)
BC4D4 does not use time delay compensation in the same way
"""

if PREPROCESSING_MODE == 'fingerflex':
    interpolated_finger_flex_cropped, db_spectrogramms_cropped = crop_for_time_delay(
        finger_processed, ecog_processed, time_delay_secs, current_fs)
    interpolated_finger_flex_val_cropped, db_spectrogramms_val_cropped = crop_for_time_delay(
        finger_processed_val, ecog_processed_val, time_delay_secs, current_fs)

    print("FingerFlex data shapes after time delay cropping:")
    print(f"  Train finger: {interpolated_finger_flex_cropped.shape}")
    print(f"  Train ECoG: {db_spectrogramms_cropped.shape}")
    print(f"  Val finger: {interpolated_finger_flex_val_cropped.shape}")
    print(f"  Val ECoG: {db_spectrogramms_val_cropped.shape}")
    
    # Assign to common variables for saving
    final_ecog_train = db_spectrogramms_cropped
    final_finger_train = interpolated_finger_flex_cropped
    final_ecog_val = db_spectrogramms_val_cropped
    final_finger_val = interpolated_finger_flex_val_cropped

elif PREPROCESSING_MODE == 'bc4d4':
    # BC4D4 doesn't use wavelet-based time delay
    # Data is already in correct format
    print("BC4D4 data shapes (no time delay cropping needed):")
    print(f"  Train ECoG: {ecog_processed.shape}")
    print(f"  Train finger: {finger_processed.shape}")
    print(f"  Val ECoG: {ecog_processed_val.shape}")
    print(f"  Val finger: {finger_processed_val.shape}")
    
    final_ecog_train = ecog_processed
    final_finger_train = finger_processed
    final_ecog_val = ecog_processed_val
    final_finger_val = finger_processed_val

In [ ]:
"""
Saving processed data
"""

import pathlib, os

SAVE_PATH = f"{pathlib.Path().resolve()}/data"

# Add suffix to distinguish between preprocessing methods
ADD_NAME = "" if PREPROCESSING_MODE == 'fingerflex' else "_bc4d4"

def save_proccessed_data(ecog_data, fingerflex_data, path, val=None, add_name="", reshape=False):
    pathlib.Path(f"{path}/train").mkdir(parents=True, exist_ok=True)
    pathlib.Path(f"{path}/val").mkdir(parents=True, exist_ok=True)
    pathlib.Path(f"{path}/test").mkdir(parents=True, exist_ok=True)
    
    ecog_path = f"{path}/train/ecog_data{add_name}.npy" if val is None else f"{path}/val/ecog_data{add_name}.npy" if \
        val is True else f"{path}/test/ecog_data{add_name}.npy"
    fingerflex_path = f"{path}/train/fingerflex_data{add_name}.npy" if val is None else f"{path}/val/fingerflex_data{add_name}.npy" if \
        val is True else f"{path}/test/fingerflex_data{add_name}.npy"
    
    if reshape:
        ecog_data = ecog_data.reshape(CHANNELS_NUM*WAVELET_NUM, -1)
    
    np.save(ecog_path, ecog_data)
    np.save(fingerflex_path, fingerflex_data)
    print(f"Saved: {ecog_path}")
    print(f"Saved: {fingerflex_path}")

print(f"Save path: {SAVE_PATH}")
print(f"File suffix: '{ADD_NAME}'")

In [ ]:
# Save initial processed data
save_proccessed_data(final_ecog_train, final_finger_train, SAVE_PATH, add_name=ADD_NAME)
save_proccessed_data(final_ecog_val, final_finger_val, SAVE_PATH, val=True, add_name=ADD_NAME)

In [ ]:
"""
Loading processed data
"""

def load_data(ecog_data_path, fingerflex_data_path):
    ecog_data = np.load(ecog_data_path)
    fingerflex_data = np.load(fingerflex_data_path)
    return ecog_data, fingerflex_data

ecog_data, fingerflex_data = load_data(
    f"{SAVE_PATH}/train/ecog_data{ADD_NAME}.npy", 
    f"{SAVE_PATH}/train/fingerflex_data{ADD_NAME}.npy"
)

ecog_data_val, fingerflex_data_val = load_data(
    f"{SAVE_PATH}/val/ecog_data{ADD_NAME}.npy", 
    f"{SAVE_PATH}/val/fingerflex_data{ADD_NAME}.npy"
)

print(f"Loaded train ECoG: {ecog_data.shape}")
print(f"Loaded train finger: {fingerflex_data.shape}")
print(f"Loaded val ECoG: {ecog_data_val.shape}")
print(f"Loaded val finger: {fingerflex_data_val.shape}")

In [12]:
fingerflex_data.shape

(5, 39980)

In [ ]:
"""
Finger motions scaling
"""

from sklearn.preprocessing import MinMaxScaler

if PREPROCESSING_MODE == 'fingerflex':
    # FingerFlex: finger data shape is (5, time)
    scaler = MinMaxScaler()
    scaler.fit(fingerflex_data.T)
    fingerflex_data_scaled = scaler.transform(fingerflex_data.T).T
    fingerflex_data_val_scaled = scaler.transform(fingerflex_data_val.T).T
    
elif PREPROCESSING_MODE == 'bc4d4':
    # BC4D4: finger data shape is (time, 5)
    # BC4D4 paper doesn't use MinMaxScaler - data is already in good range after Isolation Forest
    # But we can optionally scale if needed
    scaler = MinMaxScaler()
    scaler.fit(fingerflex_data)
    fingerflex_data_scaled = scaler.transform(fingerflex_data)
    fingerflex_data_val_scaled = scaler.transform(fingerflex_data_val)

print(f"Finger train scaled shape: {fingerflex_data_scaled.shape}")
print(f"Finger val scaled shape: {fingerflex_data_val_scaled.shape}")

In [ ]:
# Save scaled finger data
save_proccessed_data(ecog_data, fingerflex_data_scaled, SAVE_PATH, add_name=ADD_NAME)
save_proccessed_data(ecog_data_val, fingerflex_data_val_scaled, SAVE_PATH, val=True, add_name=ADD_NAME)

In [18]:
ecog_data.shape

(62, 40, 39980)

In [ ]:
"""
ECoG data scaling
"""

from sklearn.preprocessing import RobustScaler

if PREPROCESSING_MODE == 'fingerflex':
    # FingerFlex: Apply RobustScaler to wavelet spectrograms
    # Shape: (62, 40, time) -> reshape for scaling
    transformer = RobustScaler(unit_variance=True, quantile_range=(0.1, 0.9))
    transformer.fit(ecog_data.T.reshape(-1, WAVELET_NUM*CHANNELS_NUM))

    ecog_data_scaled = transformer.transform(ecog_data.T.reshape(-1, WAVELET_NUM*CHANNELS_NUM)).reshape(-1,
                                                                                    WAVELET_NUM, CHANNELS_NUM).T

    ecog_data_val_scaled = transformer.transform(ecog_data_val.T.reshape(-1, WAVELET_NUM*CHANNELS_NUM)).reshape(-1,
                                                                                    WAVELET_NUM, CHANNELS_NUM).T
    
elif PREPROCESSING_MODE == 'bc4d4':
    # BC4D4: ECoG already z-score normalized during preprocessing
    # Shape: (time, electrodes, 1) - no additional scaling needed
    # But optionally apply RobustScaler if desired
    print("BC4D4: ECoG already normalized during preprocessing")
    ecog_data_scaled = ecog_data
    ecog_data_val_scaled = ecog_data_val

print(f"ECoG train scaled shape: {ecog_data_scaled.shape}")
print(f"ECoG val scaled shape: {ecog_data_val_scaled.shape}")

In [ ]:
# Save final scaled data
save_proccessed_data(ecog_data_scaled, fingerflex_data_scaled, SAVE_PATH, add_name=ADD_NAME)
save_proccessed_data(ecog_data_val_scaled, fingerflex_data_val_scaled, SAVE_PATH, val=True, add_name=ADD_NAME)

print(f"\n{'='*60}")
print(f"PREPROCESSING COMPLETE - Mode: {PREPROCESSING_MODE}")
print(f"{'='*60}")
print(f"Final train ECoG shape: {ecog_data_scaled.shape}")
print(f"Final train finger shape: {fingerflex_data_scaled.shape}")
print(f"Final val ECoG shape: {ecog_data_val_scaled.shape}")
print(f"Final val finger shape: {fingerflex_data_val_scaled.shape}")
print(f"Files saved with suffix: '{ADD_NAME}'")